# Ankit Kryptex AI — QLoRA Fine-tuning
This notebook fine-tunes the instruction model `Qwen/Qwen2.5-0.5B-Instruct` using 4-bit QLoRA on a Colab GPU. It is not training a foundation model from random weights. The upstream model is Apache-2.0 licensed; retain its license/attribution. The resulting adapter is saved to Google Drive. A 1,000-row starter dataset is not enough to make a ChatGPT-level model; review and expand the dataset with original, high-quality examples.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, torch
REPO_URL='https://github.com/sidharthkumar8580-oss/Ankit-Kryptex-.git'
if not os.path.isdir('/content/Ankit-Kryptex-'):
    subprocess.run(['git','clone',REPO_URL,'/content/Ankit-Kryptex-'],check=True)
else:
    subprocess.run(['git','-C','/content/Ankit-Kryptex-','pull','--ff-only'],check=True)
os.chdir('/content/Ankit-Kryptex-')
assert torch.cuda.is_available(), 'GPU nahi mila. Colab Runtime > Change runtime type > T4 GPU select karo.'
print('GPU:',torch.cuda.get_device_name(0))
print('VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,1))

In [ ]:
!pip -q install -U "transformers>=4.45,<5" "datasets>=2.20" "peft>=0.13" "accelerate>=1.0" "bitsandbytes>=0.44"

In [ ]:
import json, random
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
BASE_MODEL='Qwen/Qwen2.5-0.5B-Instruct'
SYSTEM_PROMPT=('You are Ankit Kryptex AI, a helpful and conversational assistant. '
 'Reply naturally and directly. Match the user language (Hindi, Hinglish, or English). '
 'For greetings, respond warmly and briefly. Do not repeat the same canned introduction.')
rows=[]
with open('data/train.jsonl',encoding='utf-8') as f:
    for line in f:
        if line.strip():
            item=json.loads(line)
            if item.get('prompt','').strip() and item.get('response','').strip(): rows.append(item)
print('Usable examples:',len(rows))
assert len(rows)>=100, 'Dataset too small; add reviewed examples first.'
random.Random(42).shuffle(rows)
tokenizer=AutoTokenizer.from_pretrained(BASE_MODEL,trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='right'
texts=[]
for item in rows:
    messages=[{'role':'system','content':SYSTEM_PROMPT},
              {'role':'user','content':item['prompt'].strip()},
              {'role':'assistant','content':item['response'].strip()}]
    texts.append({'text':tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=False)})
dataset=Dataset.from_list(texts).train_test_split(test_size=0.1,seed=42)
print('Train:',len(dataset['train']),'Validation:',len(dataset['test']))

## Load pretrained model in 4-bit
The base model already knows general language patterns. QLoRA trains a small adapter rather than updating all 0.5B parameters, which is more practical on a free T4.

In [ ]:
compute_dtype=torch.float16
bnb_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,bnb_4bit_use_double_quant=True)
tokenizer=AutoTokenizer.from_pretrained(BASE_MODEL,trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side='right'
model=AutoModelForCausalLM.from_pretrained(BASE_MODEL,quantization_config=bnb_config,
    device_map='auto',torch_dtype=compute_dtype,trust_remote_code=True)
model.config.use_cache=False
model=prepare_model_for_kbit_training(model)
lora=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias='none',task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model=get_peft_model(model,lora)
model.print_trainable_parameters()

In [ ]:
MAX_LENGTH=1024
def tokenize(batch):
    return tokenizer(batch['text'],truncation=True,max_length=MAX_LENGTH)
tokenized=dataset.map(tokenize,batched=True,remove_columns=['text'])
collator=DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False)
OUT='/content/drive/MyDrive/AK-Byte-1-Qwen-QLoRA'
args=TrainingArguments(output_dir=OUT, num_train_epochs=3,
    per_device_train_batch_size=2,per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,learning_rate=2e-4,warmup_ratio=0.03,
    lr_scheduler_type='cosine',weight_decay=0.01,fp16=True,
    gradient_checkpointing=True,optim='paged_adamw_8bit',
    eval_strategy='epoch',save_strategy='epoch',logging_steps=10,
    save_total_limit=2,load_best_model_at_end=True,report_to='none',
    gradient_checkpointing_kwargs={'use_reentrant':False})
trainer=Trainer(model=model,args=args,train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],data_collator=collator)
print('Starting QLoRA fine-tuning…')
trainer.train()

In [ ]:
os.makedirs(OUT,exist_ok=True)
model.save_pretrained(OUT+'/adapter')
tokenizer.save_pretrained(OUT+'/adapter')
metrics=trainer.evaluate()
print('Evaluation:',metrics)
print('Adapter saved in Google Drive:',OUT+'/adapter')
print('Note: adapter is not a standalone model; inference also needs the same Qwen base model.')

In [ ]:
from peft import PeftModel
model.eval()
test_messages=[{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':'Hi'}]
inputs=tokenizer.apply_chat_template(test_messages,add_generation_prompt=True,return_tensors='pt').to(model.device)
with torch.no_grad():
    generated=model.generate(inputs,max_new_tokens=100,do_sample=True,temperature=0.7,top_p=0.9,
        repetition_penalty=1.1,pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(generated[0][inputs.shape[-1]:],skip_special_tokens=True))

## What this does and does not do
- Fine-tunes a pretrained 0.5B instruction model; it does not create a ChatGPT-scale foundation model.
- The current repository dataset has 1,000 rows and limited response diversity. Training cannot invent reliable knowledge absent from data.
- This saves a LoRA adapter to Drive. The Render API has not yet been switched to this model; serving a 0.5B model on a free 512 MB service may exceed its memory. Do not assume the old Render URL is using these new weights.
- Keep private information, passwords, API keys, and private conversations out of training data.